In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = (SparkSession.builder 
    .appName("Feature_Engineering")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math

import pandas as pd
pd.set_option('display.max_columns', None)
import plotly.express as px
import plotly.graph_objects as go
path = "/home/jovyan/work/data/bronze/"

In [12]:
window_sequential = (
    Window.partitionBy("client_id", "card_id")
    .orderBy(F.col("date").cast("long"))
)
window_24h = (
    Window.partitionBy("card_id", "weekend")
    .orderBy(F.col("date").cast("long"))
    .rangeBetween(-24 * 60 * 60, 0)
)

window_7days = (
    Window.partitionBy("card_id", "weekend")
    .orderBy(F.col("date").cast("long"))
    .rangeBetween(-7 * 24 * 60 * 60, 0)
)

Cargamos nuestro dataframe y descartamos las variables que no incorporaremos al modelo de ML/DL

In [4]:
fraud_data_df = (
    spark.read.parquet(f"{path}complete_fraud_data_df.parquet")
    .drop("mcc", "per_capita_income", "current_age",
          "card_number", "cvv", "year_pin_last_changed","card_on_dark_web",
          "address", "merchant_id", "merchant_state", "merchant_city", "zip",
    )
)

print(f"Filas: {fraud_data_df.count()}, Columnas: {len(fraud_data_df.columns)}")
fraud_data_df.printSchema()
fraud_data_df.limit(5).toPandas()

Filas: 13305915, Columnas: 28
root
 |-- id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- errors: string (nullable = true)
 |-- merchant_latitude: double (nullable = true)
 |-- merchant_longitude: double (nullable = true)
 |-- mcc_description: string (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- client_latitude: double (nullable = true)
 |-- client_longitude: double (nullable = true)
 |-- yearly_income: double (nullable = true)
 |-- total_debt: double (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- expires: date 

,id,client_id,card_id,date,amount,use_chip,errors,merchant_latitude,merchant_longitude,mcc_description,retirement_age,birth_year,birth_month,gender,client_latitude,client_longitude,yearly_income,total_debt,credit_score,num_credit_cards,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,target
0,10000109,1703,2969,2011-08-30 12:14:00,25.88,Online Transaction,None,NaN,NaN,Taxicabs and Limousines,65,1972,8,Female,39.08,-108.55,49169.0,65994.0,747,2,Mastercard,Credit,2022-06-01,True,2,10600.0,2008-05-01,False
1,10000133,1621,5036,2011-08-30 12:18:00,13.29,Swipe Transaction,None,32.9652,-117.1213,"Grocery Stores, Supermarkets",69,1953,9,Male,32.96,-117.12,74059.0,144396.0,744,4,Visa,Credit,2012-04-01,True,1,17000.0,2005-10-01,None
2,10000137,1067,1063,2011-08-30 12:19:00,10.60,Swipe Transaction,None,37.5530,-97.2549,"Grocery Stores, Supermarkets",63,1943,12,Male,37.54,-97.25,41790.0,24676.0,719,5,Visa,Credit,2023-06-01,False,1,11200.0,2005-02-01,False
3,10000175,114,3070,2011-08-30 12:26:00,46.21,Swipe Transaction,None,34.0860,-117.8843,"Artist Supply Stores, Craft Shops",65,1972,12,Female,34.18,-118.39,34441.0,907.0,725,4,Mastercard,Debit,2019-01-01,True,2,16617.0,2010-05-01,False
4,10000178,1145,6011,2011-08-30 12:26:00,78.00,Swipe Transaction,None,44.3456,-88.4343,Service Stations,65,1977,1,Female,44.26,-88.39,75080.0,35268.0,747,3,Mastercard,Debit,2024-11-01,True,1,35169.0,2010-12-01,False


In [10]:
(
    fraud_data_df
    .repartition(5, "client_id", "card_id")
    .write.mode("overwrite").parquet(f"{path}fraud_data_df")
)

Hacemos la secuencia temporal creando funciones ventana e intervalos de confianza siguiendo la distribución de Von Mises  
http://dx.doi.org/10.1016/j.eswa.2015.12.030

In [5]:
temporal_features_fraud_data_df = (
    spark.read.parquet(f"{path}fraud_data_df")
    .select("id", "client_id", "card_id", "date")
    .withColumns({
        "day_of_week": F.weekday("date"),
        "weekend": F.col("day_of_week") >= 4,
        "hour": F.hour("date"),
        "minute": F.minute("date"),
        "hour_of_day": F.col("hour") + (F.col("minute") / 60.0),
        "theta": (F.col("hour") + (F.col("minute") / 60.0)) * (2 * math.pi / 24.0),
        "sin_theta": F.sin(F.col("theta")),
        "cos_theta": F.cos(F.col("theta")),
    })
    .withColumns({
        "sum_sin_7d": F.sum("sin_theta").over(window_7days),
        "sum_cos_7d": F.sum("cos_theta").over(window_7days),
        "N_7d": F.count("*").over(window_7days),
    })
    .withColumns({
    "mu_vM_7d": 2 * F.atan(
        F.col("sum_sin_7d") / (F.sqrt(F.pow("sum_cos_7d", 2) + F.pow("sum_sin_7d", 2)) + F.col("sum_cos_7d"))
        ),
    "R_sq_7d": (F.pow(F.col("sum_sin_7d") / F.col("N_7d"), 2) + F.pow(F.col("sum_cos_7d") / F.col("N_7d"), 2)),
    })
    .withColumn("sigma_vM_7d", 
                F.sqrt(F.log(F.greatest(1.0 / F.greatest(F.col("R_sq_7d"), F.lit(1e-6)), F.lit(1.0)))),
    )
    .withColumns({
    "ic_lower_vm_90_7d": F.col("mu_vM_7d") - (1.645 * (F.col("sigma_vM_7d") / F.sqrt(F.col("N_7d")))),
    "ic_upper_vm_90_7d": F.col("mu_vM_7d") + (1.645 * (F.col("sigma_vM_7d") / F.sqrt(F.col("N_7d")))),
    "ic_lower_vm_95_7d": F.col("mu_vM_7d") - (1.960 * (F.col("sigma_vM_7d") / F.sqrt(F.col("N_7d")))),
    "ic_upper_vm_95_7d": F.col("mu_vM_7d") + (1.960 * (F.col("sigma_vM_7d") / F.sqrt(F.col("N_7d")))),
    "ic_lower_vm_99_7d": F.col("mu_vM_7d") - (2.576 * (F.col("sigma_vM_7d") / F.sqrt(F.col("N_7d")))),
    "ic_upper_vm_99_7d": F.col("mu_vM_7d") + (2.576 * (F.col("sigma_vM_7d") / F.sqrt(F.col("N_7d")))),
    })
    .withColumns({
    "is_time_in_ic_90": F.when(F.col("N_7d") == 1, True)
                         .otherwise(
                             ((F.col("theta") >= F.col("ic_lower_vm_90_7d")) & 
                              (F.col("theta") <= F.col("ic_upper_vm_90_7d"))) |
                             (((F.col("theta") + (2 * math.pi)) >= F.col("ic_lower_vm_90_7d")) & 
                              ((F.col("theta") + (2 * math.pi)) <= F.col("ic_upper_vm_90_7d")))
                         ),
    "is_time_in_ic_95": F.when(F.col("N_7d") == 1, True)
                         .otherwise(
                             ((F.col("theta") >= F.col("ic_lower_vm_95_7d")) & 
                              (F.col("theta") <= F.col("ic_upper_vm_95_7d"))) |
                             (((F.col("theta") + (2 * math.pi)) >= F.col("ic_lower_vm_95_7d")) & 
                              ((F.col("theta") + (2 * math.pi)) <= F.col("ic_upper_vm_95_7d")))
                         ),
    "is_time_in_ic_99": F.when(F.col("N_7d") == 1, True)
                        .otherwise(
                            ((F.col("theta") >= F.col("ic_lower_vm_99_7d")) & 
                             (F.col("theta") <= F.col("ic_upper_vm_99_7d"))) |
                            (((F.col("theta") + (2 * math.pi)) >= F.col("ic_lower_vm_99_7d")) & 
                             ((F.col("theta") + (2 * math.pi)) <= F.col("ic_upper_vm_99_7d")))
                        ), 
    })  
    .drop("sum_sin_7d", "hour", "minute", "sum_cos_7d", "R_sq_7d")
)

temporal_features_fraud_data_df.limit(5).toPandas()

,id,client_id,card_id,date,day_of_week,weekend,hour_of_day,theta,sin_theta,cos_theta,N_7d,mu_vM_7d,sigma_vM_7d,ic_lower_vm_90_7d,ic_upper_vm_90_7d,ic_lower_vm_95_7d,ic_upper_vm_95_7d,ic_lower_vm_99_7d,ic_upper_vm_99_7d,is_time_in_ic_90,is_time_in_ic_95,is_time_in_ic_99
0,7475412,1024,1006,2010-01-01 02:06:00,4,True,2.100000,0.549779,0.522499,0.852640,1,0.549779,1.490116e-08,0.549779,0.549779,0.549779,0.549779,0.549779,0.549779,True,True,True
1,7479155,1024,1006,2010-01-01 21:24:00,4,True,21.400000,5.602507,-0.629320,0.777146,2,-0.065450,6.364825e-01,-0.805800,0.674901,-0.947570,0.816670,-1.224807,1.093908,False,False,False
2,7481779,1024,1006,2010-01-02 15:15:00,5,True,15.250000,3.992441,-0.751840,-0.659346,3,-0.724363,1.295760e+00,-1.954999,0.506274,-2.190653,0.741928,-2.651487,1.202762,False,False,False
3,7482197,1024,1006,2010-01-02 17:05:00,5,True,17.083333,4.472406,-0.971342,-0.237686,4,-1.189935,1.189642e+00,-2.168416,-0.211454,-2.355785,-0.024086,-2.722195,0.342324,False,False,False
4,7503001,1024,1006,2010-01-08 02:05:00,4,True,2.083333,0.545415,0.518773,0.854912,5,-0.690327,1.332032e+00,-1.670258,0.289605,-1.857905,0.477251,-2.224858,0.844204,False,False,True


In [6]:
(
    temporal_features_fraud_data_df
    .drop("date")
    .repartition(6, "client_id", "card_id")
    .write.mode("overwrite").parquet(f"{path}temporal_features_fraud_data_df")
)

In [7]:
fraud_data_df = (
    spark.read.parquet(f"{path}fraud_data_df")
    .select("id", "client_id", "card_id", "date", "amount")
)

temporal_features_fraud_data_df = (
    spark.read.parquet(f"{path}temporal_features_fraud_data_df")
    .select("id", "client_id", "card_id","weekend", "N_7d")
)

fraud_data_2_df = (
    temporal_features_fraud_data_df
    .join(fraud_data_df, on=["id", "client_id", "card_id"], how="left")
)

fraud_data_2_df.limit(5).toPandas()

,id,client_id,card_id,weekend,N_7d,date,amount
0,12813965,1262,2432,False,3,2013-05-29 13:30:00,0.94
1,17627304,177,2681,True,4,2016-04-08 10:33:00,46.96
2,17635265,177,2681,True,7,2016-04-10 07:35:00,100.00
3,7475329,1129,102,True,1,2010-01-01 00:02:00,80.00
4,7477312,1288,1013,True,1,2010-01-01 12:36:00,11.52


In [17]:
amount_features_fraud_data_df = (
    fraud_data_2_df
    .withColumns({
        "is_refund": F.col("amount") < 0.0,
        "log_amount": F.log(F.abs(F.col("amount")) + 1.0)
    })
    
    .withColumns({
        "mean_log_amount_7d": F.avg("log_amount").over(window_7days),
        "stddev_log_amount_7d": F.coalesce(F.stddev("log_amount").over(window_7days), F.lit(0.0)),
        "sum_amount_24h": F.sum("amount").over(window_24h),
        "sum_amount_7d": F.sum("amount").over(window_7days)
    })
    .withColumns({
        "daily_spend_rate_7d": F.when(F.col("weekend"), F.col("sum_amount_7d") / 3.0)
                                .otherwise(F.col("sum_amount_7d") / 4.0 ),
        "acceleration_ratio_24h_vs_7d": F.coalesce(F.col("sum_amount_24h") / F.col("sum_amount_7d"), F.lit(0.0))
    })
    .withColumns({        
        "ic_lower_amt_90_7d": F.col("mean_log_amount_7d") - (1.645 * F.col("stddev_log_amount_7d")),
        "ic_upper_amt_90_7d": F.col("mean_log_amount_7d") + (1.645 * F.col("stddev_log_amount_7d")),    
        "ic_lower_amt_95_7d": F.col("mean_log_amount_7d") - (1.960 * F.col("stddev_log_amount_7d")),
        "ic_upper_amt_95_7d": F.col("mean_log_amount_7d") + (1.960 * F.col("stddev_log_amount_7d")),
        "ic_lower_amt_99_7d": F.col("mean_log_amount_7d") - (2.576 * F.col("stddev_log_amount_7d")),
        "ic_upper_amt_99_7d": F.col("mean_log_amount_7d") + (2.576 * F.col("stddev_log_amount_7d"))
    })
    .withColumns({
        "is_amt_in_ic_90": F.when(F.col("N_7d") == 1, True)
                            .otherwise(
                                (F.col("log_amount") >= F.col("ic_lower_amt_90_7d")) & 
                                (F.col("log_amount") <= F.col("ic_upper_amt_90_7d"))
                            ),
        "is_amt_in_ic_95": F.when(F.col("N_7d") == 1, True)
                            .otherwise(
                                (F.col("log_amount") >= F.col("ic_lower_amt_95_7d")) & 
                                (F.col("log_amount") <= F.col("ic_upper_amt_95_7d"))
                            ),
        "is_amt_in_ic_99": F.when(F.col("N_7d") == 1, True)
                            .otherwise(
                                (F.col("log_amount") >= F.col("ic_lower_amt_99_7d")) & 
                                (F.col("log_amount") <= F.col("ic_upper_amt_99_7d"))
                            )
    })  
    .drop("weekend", "log_amount", "N_7d")
)

amount_features_fraud_data_df.limit(5).toPandas()

,id,client_id,card_id,date,amount,is_refund,mean_log_amount_7d,stddev_log_amount_7d,sum_amount_24h,sum_amount_7d,daily_spend_rate_7d,acceleration_ratio_24h_vs_7d,ic_lower_amt_90_7d,ic_upper_amt_90_7d,ic_lower_amt_95_7d,ic_upper_amt_95_7d,ic_lower_amt_99_7d,ic_upper_amt_99_7d,is_amt_in_ic_90,is_amt_in_ic_95,is_amt_in_ic_99
0,7488762,1786,1000,2010-01-04 11:41:00,15.79,False,2.820783,0.000000,15.79,15.79,3.9475,1.00000,2.820783,2.820783,2.820783,2.820783,2.820783,2.820783,True,True,True
1,7489883,1786,1000,2010-01-04 15:56:00,31.72,False,3.154385,0.471784,47.51,47.51,11.8775,1.00000,2.378301,3.930469,2.229689,4.079081,1.939070,4.369700,True,True,True
2,7491147,1786,1000,2010-01-05 06:03:00,5.86,False,2.744826,0.783904,53.37,53.37,13.3425,1.00000,1.455303,4.034349,1.208373,4.281278,0.725488,4.764164,True,True,True
3,7491738,1786,1000,2010-01-05 08:22:00,2.68,False,2.384348,0.964079,56.05,56.05,14.0125,1.00000,0.798437,3.970258,0.494752,4.273943,-0.099121,4.867816,True,True,True
4,7500970,1786,1000,2010-01-07 12:44:00,18.44,False,2.500945,0.874678,18.44,74.49,18.6225,0.24755,1.062100,3.939789,0.786576,4.215313,0.247775,4.754114,True,True,True


In [18]:
(
    amount_features_fraud_data_df
    .drop("date", "amount")
    .repartition(6, "client_id", "card_id")
    .write.mode("overwrite").parquet(f"{path}amount_features_fraud_data_df")
)

In [21]:
velocity_features_fraud_data_df = (
    fraud_data_2_df
    .drop("amount")
    .withColumns({
        "prev_date_sec": F.lag(F.col("date").cast("long"), 1).over(window_sequential)
    })
    .withColumns({
        "mins_since_last_tx": (F.col("date").cast("long") - F.col("prev_date_sec")) / 60.0,
        "log_mins_since_last_tx": F.log(F.coalesce(F.col("mins_since_last_tx"), F.lit(0.0)) + 1.0)
    })
    .withColumns({
        "mean_log_mins_7d": F.avg("log_mins_since_last_tx").over(window_7days),
        "stddev_log_mins_7d": F.coalesce(F.stddev("log_mins_since_last_tx").over(window_7days), F.lit(0.0))
    })
    .withColumns({
        "ic_lower_mins_90_7d": F.col("mean_log_mins_7d") - (1.645 * F.col("stddev_log_mins_7d")),
        "ic_upper_mins_90_7d": F.col("mean_log_mins_7d") + (1.645 * F.col("stddev_log_mins_7d")),
        "ic_lower_mins_95_7d": F.col("mean_log_mins_7d") - (1.960 * F.col("stddev_log_mins_7d")),
        "ic_upper_mins_95_7d": F.col("mean_log_mins_7d") + (1.960 * F.col("stddev_log_mins_7d")),
        "ic_lower_mins_99_7d": F.col("mean_log_mins_7d") - (2.576 * F.col("stddev_log_mins_7d")),
        "ic_upper_mins_99_7d": F.col("mean_log_mins_7d") + (2.576 * F.col("stddev_log_mins_7d")),
    })
    .withColumns({
        "is_speed_in_ic_90": F.when(F.col("N_7d") == 1, True)
                              .otherwise(
                                  (F.col("log_mins_since_last_tx") >= F.col("ic_lower_mins_90_7d")) & 
                                  (F.col("log_mins_since_last_tx") <= F.col("ic_upper_mins_90_7d"))
                              ),
        "is_speed_in_ic_95": F.when(F.col("N_7d") == 1, True)
                              .otherwise(
                                  (F.col("log_mins_since_last_tx") >= F.col("ic_lower_mins_95_7d")) & 
                                  (F.col("log_mins_since_last_tx") <= F.col("ic_upper_mins_95_7d"))
                              ),
        "is_speed_in_ic_99": F.when(F.col("N_7d") == 1, True)
                              .otherwise(
                                  (F.col("log_mins_since_last_tx") >= F.col("ic_lower_mins_99_7d")) & 
                                  (F.col("log_mins_since_last_tx") <= F.col("ic_upper_mins_99_7d"))
                              )
    })
    .drop("weekend", "N_7d", "prev_date_sec", "mins_since_last_tx")
)

velocity_features_fraud_data_df.limit(5).toPandas()

,id,client_id,card_id,date,log_mins_since_last_tx,mean_log_mins_7d,stddev_log_mins_7d,ic_lower_mins_90_7d,ic_upper_mins_90_7d,ic_lower_mins_95_7d,ic_upper_mins_95_7d,ic_lower_mins_99_7d,ic_upper_mins_99_7d,is_speed_in_ic_90,is_speed_in_ic_95,is_speed_in_ic_99
0,7487228,1783,10,2010-01-04 06:00:00,7.150701,7.150701,0.000000,7.150701,7.150701,7.150701,7.150701,7.150701,7.150701,True,True,True
1,7487382,1783,10,2010-01-04 06:38:00,3.663562,5.407132,2.465780,1.350923,9.463340,0.574202,10.240061,-0.944718,11.758981,True,True,True
2,7487388,1783,10,2010-01-04 06:40:00,1.098612,3.970958,3.037732,-1.026111,8.968028,-1.982996,9.924913,-3.854239,11.796156,True,True,True
3,7487468,1783,10,2010-01-04 07:00:00,3.044522,3.739349,2.523182,-0.411285,7.889984,-1.206087,8.684786,-2.760368,10.239067,True,True,True
4,7487889,1783,10,2010-01-04 08:35:00,4.564348,3.904349,2.216069,0.258916,7.549782,-0.439145,8.247844,-1.804244,9.612942,True,True,True


In [22]:
(
    velocity_features_fraud_data_df
    .drop("date")
    .repartition(6, "client_id", "card_id")
    .write.mode("overwrite").parquet(f"{path}velocity_features_fraud_data_df")
)